In [1]:
!nvidia-smi

Sat May 16 23:28:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 1. 구글 드라이브 연동 (마운트)
from google.colab import drive
drive.mount('/content/drive')

import sys
import os
import pandas as pd

# 2. 파이썬 파일들이 있는 드라이브 경로 설정 및 시스템 패스에 추가
project_path = '/content/drive/MyDrive/GCY6902_01_TFT/005.regression_final/'
sys.path.append(project_path)

from inference import *

file_path = os.path.join(project_path, 'data/raw/binance_processed_futures_8hr_260514.parquet')
df = pd.read_parquet(file_path)
var_list = ['Open', 'High', 'Low', 'Close', 'Volume'
            , 'Number of trades'
            , 'fundingRate']
df['BAS_DT']=df['Open time'].dt.strftime('%Y%m%d%H')
df = df.loc[df['BAS_DT']>='20260328']
df['day_of_week'] = df['Open time'].dt.day_of_week # 요일 0:월, 1:화, ... 6:일
df['hour'] = df['Open time'].dt.hour/8 # 시간대(0,1,2)
var_time = ['BAS_DT','day_of_week','hour']

Mounted at /content/drive


In [6]:
# 테스트하고 싶은 특정 실험의 SEQ 설정
CHOSEN_SEQ = 21  # 21, 42, 63, 84 중 현재 평가할 폴더의 하이퍼파라미터 입력

best_config = {
    'd_model': 32,
    'hidden_dim': 64,
    'lstm_hidden': 64,
    'n_heads': 8,
    'dropout': 0.2,
    'past_vars': len(var_list), # run_regression.py 스크립트 내 변수 개수 연동
    'known_vars': 1,
    'static_vars': 1,
    'output_mode': "regression"
}

completed_outdir = project_path + "outputs/dl/1_2605160733_D32_H64_N8_SEQ21/"

# 평가 엔진 초기화
evaluator = TFTDirectTrajectoryEvaluator(model_dir=completed_outdir, config_dict=best_config)

# [수정 1] 인퍼런스 엔진의 개편된 3개 아웃풋 스펙에 맞춰 리턴 변수 리시빙 처리
strategy_report, trajectory_df, xai_dict = evaluator.evaluate_weekly_strategy(
    df_raw=df,
    seq_length=CHOSEN_SEQ,  
    base_dt_str='2026040500',   
    target_dt_str='2026041200', 
    threshold=0.03              
)
print(xai_dict)

# 전체 요약 성과 출력
print("📊 [TFT 모형 기반 전종목 스윙 전략 마스터 요약 리포트]")
display(strategy_report)

# =========================================================================
# 3. 포트폴리오 백테스트 성과 분석 및 알파(Alpha) 검증 (코인 5개 전체 대응형)
# =========================================================================
# 매수 시그널이 켜진 종목 필터링
buy_portfolio = strategy_report[strategy_report['Strategy_Signal'] == 'BUY']

if not buy_portfolio.empty:
    # [수정 2] head(5) 제거: 탑 5개가 아닌 시그널이 켜진 대상 자산 전체를 포트폴리오로 채택
    portfolio_expected = buy_portfolio['Expected_Return_Pct'].mean()
    portfolio_actual_strat = buy_portfolio['Actual_Strategy_Return_Pct'].mean()
    portfolio_actual_hold = buy_portfolio['Passive_Market_Return_Pct'].mean()
    
    print("\n==================================================================")
    print("🎯 [전략 포트폴리오 백테스트 정산 리포트]")
    print("==================================================================")
    print(f"🔹 선정 포트폴리오 자산: {list(buy_portfolio['Symbol'])}")
    print(f"📈 모델 궤적상 최적 스윙 예상 수익률     : {portfolio_expected:.2f}%")
    print(f"💰 해당 타이밍 실전 진입 시 실제 수익률  : {portfolio_actual_strat:.2f}%")
    print(f"💤 동기간 단순 매수 후 방치 시 실제 수익률: {portfolio_actual_hold:.2f}%")
    print(f"🚀 타이밍 최적화를 통한 주간 초과 알파   : {portfolio_actual_strat - portfolio_actual_hold:.2f}%p")
    print("==================================================================\n")
    
    # -----------------------------------------------------------------
    # [Streamlit 시각화 연동용] 최종 아웃풋 파일 자동 백업 로직 추가
    # -----------------------------------------------------------------
    # 대시보드가 읽어갈 수 있도록 파일명 규격 통일하여 엑셀 저장
    strategy_report.to_excel(completed_outdir + "Backtest_Report_0405_to_0412.xlsx", index=False)
    trajectory_df.to_excel(completed_outdir + "Trajectory_Detail_0405_to_0412.xlsx", index=False)
    
    # 구조화된 딕셔너리 형태인 XAI 매트릭스는 피클 파일로 바이너리 백업
    with open(os.path.join(completed_outdir, 'xai_weights_map.pkl'), 'wb') as f:
        pickle.dump(xai_dict, f)
        
    print(f"💾 대시보드 시각화용 아웃풋 파일 아티팩트 저장 완료 ➡️ 경로: {completed_outdir}")
else:
    print(f"\n⚠️ 임계치({threshold*100}%)를 넘는 매수 추천 자산이 존재하지 않습니다.")

✅ Direct Inference Engine 로드 완료


Evaluating Symbols: 100%|██████████| 5/5 [00:01<00:00,  2.75it/s]

{'BTCUSDT':             Feature  Importance
0            Volume    0.684031
1              Open    0.082952
2             Close    0.065481
3       fundingRate    0.063945
4              High    0.054193
5               Low    0.024740
6  Number of trades    0.024659, 'DOGEUSDT':             Feature  Importance
0            Volume    0.392783
1       fundingRate    0.173969
2  Number of trades    0.165123
3              High    0.130556
4              Open    0.081735
5               Low    0.041735
6             Close    0.014098, 'ETHUSDT':             Feature  Importance
0            Volume    0.464428
1       fundingRate    0.159161
2              High    0.128666
3  Number of trades    0.103338
4              Open    0.081159
5               Low    0.050106
6             Close    0.013142, 'SOLUSDT':             Feature  Importance
0            Volume    0.428173
1              High    0.159269
2       fundingRate    0.156539
3  Number of trades    0.105272
4              Open    

,Symbol,Current_Price(4/5),Predicted_Min_Price,Predicted_Max_Price,Best_Buy_Timing,Best_Sell_Timing,Expected_Return_Pct,Actual_Strategy_Return_Pct,Passive_Market_Return_Pct,Strategy_Signal
0,XRPUSDT,1.31170,0.106354,0.116759,T+1,T+19,9.784083,3.566202,3.156209,BUY
1,DOGEUSDT,0.09174,0.065114,0.069667,T+12,T+20,6.992672,1.434471,1.384347,BUY
2,ETHUSDT,2061.27000,1836.251896,1848.243831,T+9,T+21,0.653066,2.202359,10.832642,HOLD
3,SOLUSDT,80.65000,101.672578,102.021975,T+5,T+20,0.343650,4.131927,5.207688,HOLD
4,BTCUSDT,67177.00000,51206.923660,51265.614084,T+12,T+21,0.114614,3.192755,8.725903,HOLD



🎯 [전략 포트폴리오 백테스트 정산 리포트]
🔹 선정 포트폴리오 자산: ['XRPUSDT', 'DOGEUSDT']
📈 모델 궤적상 최적 스윙 예상 수익률     : 8.39%
💰 해당 타이밍 실전 진입 시 실제 수익률  : 2.50%
💤 동기간 단순 매수 후 방치 시 실제 수익률: 2.27%
🚀 타이밍 최적화를 통한 주간 초과 알파   : 0.23%p

💾 대시보드 시각화용 아웃풋 파일 아티팩트 저장 완료 ➡️ 경로: /content/drive/MyDrive/GCY6902_01_TFT/005.regression_final/outputs/dl/1_2605160733_D32_H64_N8_SEQ21/
